# Revarie LM v1.0 – Nightly Memory Consolidation
This notebook runs daily at 1:00 AM IST to consolidate the previous day's conversations into summary vectors and update persona LoRA adapters.

**Theoretical foundation:**
- Tulving (1972): Episodic memory consolidation during sleep.
- Stickgold & Walker (2013): Sleep-dependent memory triage.
- McClelland et al. (1995): Complementary learning systems theory.

In [ ]:
!pip install -q numpy aiohttp cloudflare python-dotenv sentence-transformers

import os
import json
import asyncio
import hashlib
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
import aiohttp

# Kaggle secrets (set these in notebook settings)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

VAULT_API_URL = user_secrets.get_secret("VAULT_API_URL")
VAULT_API_KEY = user_secrets.get_secret("VAULT_API_KEY")
VECTORIZE_API_URL = user_secrets.get_secret("VECTORIZE_API_URL")
VECTORIZE_API_TOKEN = user_secrets.get_secret("VECTORIZE_API_TOKEN")
CLOUDFLARE_ACCOUNT_ID = user_secrets.get_secret("CLOUDFLARE_ACCOUNT_ID")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("✅ Secrets loaded")

In [ ]:
@dataclass
class ParticipantSession:
    participant_id: str
    name: str
    study_group: str
    day_number: int
    session_date: str
    pre_vams: Dict[str, int]
    post_vams: Dict[str, int]
    chat_messages: List[Dict[str, Any]]

async def fetch_participant_sessions(target_date: str) -> List[ParticipantSession]:
    """Fetch all participant sessions for a given date from D1 via Vault API."""
    headers = {
        "x-api-key": VAULT_API_KEY,
        "Content-Type": "application/json"
    }
    
    # Get all participants who had a session on target_date
    url = f"{VAULT_API_URL}/sessions/{target_date}"
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                print(f"Error fetching sessions: {resp.status}")
                return []
            data = await resp.json()
            
    sessions = []
    for item in data.get("sessions", []):
        sess = ParticipantSession(
            participant_id=item["participant_id"],
            name=item.get("name", ""),
            study_group=item.get("study_group", "A"),
            day_number=item["day_number"],
            session_date=item["session_date_ist"],
            pre_vams=item.get("pre_vams", {}),
            post_vams=item.get("post_vams", {}),
            chat_messages=item.get("chat_messages", [])
        )
        sessions.append(sess)
    
    print(f"📊 Fetched {len(sessions)} sessions for {target_date}")
    return sessions

In [ ]:
async def generate_summary(session: ParticipantSession) -> str:
    """Generate a daily summary using a lightweight LLM (Groq)."""
    groq_api_key = user_secrets.get_secret("GROQ_API_KEY")
    
    # Build conversation string
    convo = "\n".join([
        f"{msg['role']}: {msg['content']}" for msg in session.chat_messages[:50]
    ])
    
    # Persona-specific prompt
    if session.study_group == "A":
        prompt = f"""You are Samara, a warm and empathetic AI companion.\n"""\
                 f"""Create a personal summary of today's conversation with {session.name}.\n"""\
                 f"""Use their name naturally. Note their emotions and personal details shared.\n"""\
                 f"""Conversation:\n{convo}\n\nSummary:"""
    else:
        prompt = f"""You are Artery 1.0, a precise and functional AI assistant.\n"""\
                 f"""Create a factual summary of today's interaction with participant {session.participant_id}.\n"""\
                 f"""Be neutral. Record only factual information.\n"""\
                 f"""Conversation:\n{convo}\n\nSummary:"""

    async with aiohttp.ClientSession() as client:
        headers = {
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.3,
            "max_tokens": 500
        }
        async with client.post("https://api.groq.com/openai/v1/chat/completions",
                               headers=headers, json=payload) as resp:
            data = await resp.json()
            return data["choices"][0]["message"]["content"]


In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model (cached)
model = SentenceTransformer('all-MiniLM-L6-v2')

async def upsert_summary_vector(session: ParticipantSession, summary: str):
    """Generate embedding and upsert to Cloudflare Vectorize."""
    embedding = model.encode(summary, normalize_embeddings=True).tolist()
    
    vector_id = f"{session.participant_id}_{session.day_number}_summary_{hashlib.md5(summary.encode()).hexdigest()[:8]}"
    
    payload = {
        "namespace": f"participant_{session.participant_id}",
        "vectors": [{
            "id": vector_id,
            "values": embedding,
            "metadata": {
                "participant_id": session.participant_id,
                "day_number": session.day_number,
                "memory_type": "daily_summary",
                "session_date": session.session_date,
                "study_group": session.study_group,
                "summary_text": summary[:200]
            }
        }]
    }
    
    url = f"{VECTORIZE_API_URL}/upsert"
    headers = {
        "Authorization": f"Bearer {VECTORIZE_API_TOKEN}",
        "Content-Type": "application/json"
    }
    
    async with aiohttp.ClientSession() as client:
        async with client.post(url, headers=headers, json=payload) as resp:
            if resp.status == 200:
                print(f"  ✅ Upserted summary for {session.participant_id}")
            else:
                print(f"  ❌ Failed upsert for {session.participant_id}: {resp.status}")


In [ ]:
async def update_d1_summary(session: ParticipantSession, summary: str):
    """Store summary text in D1 for quick retrieval."""
    url = f"{VAULT_API_URL}/session/summary"
    headers = {
        "x-api-key": VAULT_API_KEY,
        "Content-Type": "application/json"
    }
    payload = {
        "participant_id": session.participant_id,
        "day_number": session.day_number,
        "summary": summary
    }
    
    async with aiohttp.ClientSession() as client:
        async with client.post(url, headers=headers, json=payload) as resp:
            if resp.status == 200:
                print(f"  📝 Updated D1 for {session.participant_id}")
            else:
                print(f"  ⚠️ D1 update failed: {resp.status}")


In [ ]:
async def main():
    # Target date: yesterday in IST
    ist_now = datetime.utcnow() + timedelta(hours=5, minutes=30)
    yesterday = (ist_now - timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"🔄 Running consolidation for {yesterday}")
    
    sessions = await fetch_participant_sessions(yesterday)
    
    if not sessions:
        print("⚠️ No sessions found for yesterday. Exiting.")
        return
    
    for sess in sessions:
        print(f"\n👤 Processing {sess.participant_id} ({sess.name}) - Day {sess.day_number}")
        try:
            summary = await generate_summary(sess)
            print(f"  📋 Summary: {summary[:100]}...")
            
            await upsert_summary_vector(sess, summary)
            await update_d1_summary(sess, summary)
            
        except Exception as e:
            print(f"  ❌ Error processing {sess.participant_id}: {e}")
    
    print(f"\n✅ Consolidation complete for {yesterday}")

if __name__ == "__main__":
    await main()

## Keep-Alive (Prevent Kaggle Timeout)
The following cell runs a background thread that prints a timestamp every 5 minutes to keep the session alive during long consolidation runs.

In [ ]:
import threading
import time
from datetime import datetime

def keep_alive():
    while True:
        time.sleep(300)
        print(f"⏰ Keep-alive ping at {datetime.utcnow().isoformat()}")

threading.Thread(target=keep_alive, daemon=True).start()